# ✅ Corrected EEG Preprocessing for CTNet
This notebook builds fixed-length CTNet-ready EEG trials using **your CSV structure**, with:
- Correct MI trial segmentation
- Correct bandpass filtering
- Correct normalization
- Correct 2-second (500-sample) resampling
- Correct label encoding
- Correct trial-level train/test split
- Saves `X_train_ctnet.npy`, `y_train_ctnet.npy`, `X_test_ctnet.npy`, `y_test_ctnet.npy`

In [9]:
import pandas as pd
import numpy as np
from scipy.signal import butter, filtfilt, resample
from sklearn.preprocessing import StandardScaler, LabelEncoder

files = [
    'data/EEG_bar_trials_20_classes_3_20251118_164048.csv',
    'data/EEG_bar_trials_20_classes_3_20251118_174039.csv',
    'data/EEG_bar_trials_20_classes_3_20251118_174039.csv',
    'data/EEG_bar_trials_30_classes_3_20251118_161525.csv'
]

data = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
MI = data[data['phase']=='stimulus'].reset_index(drop=True)

signals = MI.iloc[:,5:13].values
labels_raw = MI['class_label'].values
splits_raw = MI['split'].values

print('MI rows:', MI.shape)
print('Raw labels:', np.unique(labels_raw))

MI rows: (272274, 22)
Raw labels: [1 2 3]


In [10]:
def bandpass(sig, low=4, high=40, fs=250, order=4):
    nyq = 0.5*fs
    b,a = butter(order, [low/nyq, high/nyq], btype='band')
    return filtfilt(b,a,sig)

fs = 250
filtered = np.zeros_like(signals)
for ch in range(8):
    filtered[:,ch] = bandpass(signals[:,ch], fs=fs)

scaler = StandardScaler()
eeg_norm = scaler.fit_transform(filtered)

print('Filtered + normalized EEG:', eeg_norm.shape)

Filtered + normalized EEG: (272274, 8)


In [11]:
trials=[]; trial_labels=[]; trial_splits=[]
cur_label = labels_raw[0]
cur_split = splits_raw[0]
current=[]

for i in range(len(labels_raw)):
    if labels_raw[i] != cur_label:
        trials.append(np.array(current))
        trial_labels.append(cur_label)
        trial_splits.append(cur_split)
        current=[]
        cur_label = labels_raw[i]
        cur_split = splits_raw[i]
    current.append(eeg_norm[i])

if len(current)>0:
    trials.append(np.array(current))
    trial_labels.append(cur_label)
    trial_splits.append(cur_split)

print('Extracted trials:', len(trials))
print('Trial labels:', np.unique(trial_labels))

Extracted trials: 192
Trial labels: [1 2 3]


In [12]:
le = LabelEncoder()
trial_labels_enc = le.fit_transform(trial_labels)
print('Label mapping:', dict(zip(le.classes_, le.transform(le.classes_))))

Label mapping: {np.int64(1): np.int64(0), np.int64(2): np.int64(1), np.int64(3): np.int64(2)}


In [ ]:
target_len = fs*2
X=[]; y=[]; final_splits=[]

for tr, lbl, sp in zip(trials, trial_labels_enc, trial_splits):
    if len(tr)<40: continue
    tr_r = resample(tr, target_len)
    tr_r = (tr_r - tr_r.mean(axis=0))/(tr_r.std(axis=0)+1e-8)
    X.append(tr_r); y.append(lbl); final_splits.append(sp)

X=np.array(X); y=np.array(y); final_splits=np.array(final_splits)

print('Final trial dataset:', X.shape, y.shape)

Final trial dataset: (192, 250, 8) (192,)


In [14]:
X_ctnet = np.transpose(X,(0,2,1))
print('CTNet shape:', X_ctnet.shape)

CTNet shape: (192, 8, 250)


In [15]:
train_mask = final_splits=='train'
test_mask  = final_splits=='test'

X_train = X_ctnet[train_mask]
y_train = y[train_mask]
X_test  = X_ctnet[test_mask]
y_test  = y[test_mask]

print('Train:', X_train.shape, np.unique(y_train))
print('Test:', X_test.shape, np.unique(y_test))

Train: (156, 8, 250) [0 1 2]
Test: (36, 8, 250) [0 1 2]


In [8]:
np.save('X_train_ctnet.npy', X_train)
np.save('y_train_ctnet.npy', y_train)
np.save('X_test_ctnet.npy', X_test)
np.save('y_test_ctnet.npy', y_test)
print('Saved all CTNet .npy files!')

Saved all CTNet .npy files!
